In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

#configration
model_id = "Qwen/Qwen2-0.5B-instruct"

#load model and tokenzer
print(f"loading model {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
    )
# create HF pipline
pip = pipeline(
    "text_generation",
    model=model,
    tokenizer = tokenizer,
    max_new_tokens=128,
    temperature =0.1,
    do_sample = True
)
# langchain llm wrapper
raw_llm = HuggingFacePipeline(pipeline=pip)

# define the qwen chat formate
def qwen_chat_format(input_dict):
    messages = [
        {
            "role":"system",
            "content": """
            you are helpfull AI assistant , answer questions strickly provide on context.
            do not use any external knowledge or makup infromation.
            if answer not in context respond exectly with "I don't know" and do not provide any other information.
            Always start your response with "Answer ": flowed by the answer or "i don't know" if answer not in context.
            """
        },
        {
            "role":"user",
            "content":f"""
            context:{input_dict["context"]}
            question:{input_dict["question"]}
            """
        }  
    ]
    formatted_prompt =tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return formatted_prompt
# generation function
def gereration_with_Qwen(formatted_prompt):
    response = raw_llm(formatted_prompt)
    
    generated = response.split(formatted_prompt)[-1].strip()
    if generated.startswith("Answer:"):
        generated = generated.split("Answer:", 1)[1].strip()
    return generated
# create llm chain with formatting
llm_with_format = (
    RunnableLambda(qwen_chat_format)
    | RunnableLambda(gereration_with_Qwen)
    | StrOutputParser()
)
print("llm create successfully")

c:\Users\Ahsan Ali\Desktop\AI-course\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loading model Qwen/Qwen2-0.5B-instruct...


c:\Users\Ahsan Ali\Desktop\AI-course\.venv-1\Lib\site-packages\huggingface_hub\file_download.py:747: UserWarning: Not enough free disk space to download the file. The expected file size is: 988.10 MB. The target location C:\Users\Ahsan Ali\.cache\huggingface\hub\models--Qwen--Qwen2-0.5B-instruct\blobs only has 418.14 MB free disk space.
  warnings.warn(
c:\Users\Ahsan Ali\Desktop\AI-course\.venv-1\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ahsan Ali\.cache\huggingface\hub\models--Qwen--Qwen2-0.5B-instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Wi

KeyError: "Unknown task text_generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"